In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/common.ipynb").is_file())
get_ipython().run_line_magic("run", '"' + (ROOT / "src/common.ipynb").as_posix() + '"')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import box
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter


In [ ]:
monthly_burden = pd.read_csv(DATA / "monthly_burden_means.csv", dtype={"GEOID": str}, float_precision="round_trip")
monthly_burden["GEOID"] = monthly_burden.GEOID.str.zfill(5)
assert not monthly_burden.duplicated(["GEOID", "MONTH"]).any()
regions = ["National", "South", "West", "Central", "Northeast"]
months = {1: "January", 4: "April", 7: "July", 10: "October"}
master_geoids = set(load_data("SP").GEOID)
assert set(monthly_burden.GEOID) == master_geoids
assert set(monthly_burden.MONTH) == set(months)
assert len(monthly_burden) == len(master_geoids) * len(months)
trends = monthly_burden.dropna(subset=["early_pct", "late_pct"]).copy()
trends["region"] = county_regions(trends.GEOID)
trends["change_pp"] = trends.late_pct - trends.early_pct
assert set(trends.MONTH) == set(months)
assert np.isfinite(trends[["early_pct", "late_pct", "change_pp"]].to_numpy()).all()


In [ ]:
samples = {code: load_data(code) for code in MODELS}
for frame in samples.values():
    frame["region"] = county_regions(frame.GEOID)
level_rows = []
for region in regions:
    regional = {
        code: frame if region == "National" else frame.loc[frame.region.eq(region)]
        for code, frame in samples.items()
    }
    count = regional["ALL"].GEOID.nunique()
    assert all(frame.GEOID.nunique() == count for frame in regional.values())
    values = {SEASON_NAMES[code]: frame.EB_pct.mean() for code, frame in regional.items()}
    seasonal_values = [values[SEASON_NAMES[code]] for code in SEASONS]
    level_rows.append({"Region": region, "Counties": count,
        **{f"{period} (%)": value for period, value in values.items()},
        "Seasonal range (pp)": max(seasonal_values) - min(seasonal_values)})
table_s3 = pd.DataFrame(level_rows)
export_table("TableS3", table_s3)
display(table_s3.round(2))


In [ ]:
change_rows = []
for region in regions:
    for month, month_name in months.items():
        data = trends.loc[trends.MONTH.eq(month)]
        if region != "National":
            data = data.loc[data.region.eq(region)]
        change_rows.append({"Region": region, "Month": month_name,
            "2009–2013 (%)": data.early_pct.mean(), "2018–2022 (%)": data.late_pct.mean(),
            "Mean change (pp)": data.change_pp.mean(),
            "Mean absolute change (pp)": data.change_pp.abs().mean()})
table_s4 = pd.DataFrame(change_rows)
export_table("TableS4", table_s4)
display(table_s4.round(2))


In [ ]:
def map_geometry(name):
    data = load_geo(name).to_crs(4326)
    data = data.loc[data.geometry.intersects(box(-125.0, 24.0, -66.5, 49.5))].copy()
    return data.to_crs(5070)

counties, states = map_geometry("counties"), map_geometry("states")
counties["GEOID"] = counties.GEOID.astype(str).str.zfill(5)
vmin, vmax = trends.early_pct.min(), trends.early_pct.max()
max_difference = trends.change_pp.abs().max()
norm = TwoSlopeNorm(vmin=-max_difference, vcenter=0, vmax=max_difference)
plt.rcdefaults()
plt.rcParams.update({"svg.fonttype": "none", "pdf.fonttype": 42})
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
month_labels = {1: "Jan.", 4: "Apr.", 7: "Jul.", 10: "Oct."}
for row, month in enumerate(months):
    data = counties.merge(trends.loc[trends.MONTH.eq(month)], on="GEOID", how="left", validate="one_to_one")
    for column in range(2):
        ax = axes[row, column]
        ax.set_axis_off()
        if column == 0:
            data.plot(column="early_pct", ax=ax, cmap="viridis", vmin=vmin, vmax=vmax, linewidth=0, rasterized=True)
        else:
            data.plot(column="change_pp", ax=ax, cmap="RdBu_r", norm=norm, linewidth=0, rasterized=True)
        states.boundary.plot(ax=ax, color="black", linewidth=0.5, alpha=0.9)
        label = f"{month_labels[month]} 2009–2013" if column == 0 else f"{month_labels[month]} Difference"
        ax.set_title(label, fontsize=14, pad=10)
        ax.text(0, 0.99, "abcdefgh"[row * 2 + column], transform=ax.transAxes,
                fontsize=20, fontweight="bold", va="top")
level_map = ScalarMappable(cmap="viridis")
level_map.set_clim(vmin, vmax)
change_map = ScalarMappable(cmap="RdBu_r", norm=norm)
left = fig.add_axes([0.15, 0.08, 0.3, 0.015])
right = fig.add_axes([0.55, 0.08, 0.3, 0.015])
for colorbar in [fig.colorbar(level_map, cax=left, orientation="horizontal", label="Energy Burden (%)"),
                 fig.colorbar(change_map, cax=right, orientation="horizontal", label="Difference in Energy Burden (percentage points)")]:
    colorbar.ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
plt.subplots_adjust(hspace=0.05, wspace=0.02, bottom=0.12)
fig.savefig(OUT / "figures/Figure1.png", dpi=900, bbox_inches="tight")
fig.savefig(OUT / "figures/Figure1.pdf", dpi=900, bbox_inches="tight")
plt.show()
plt.close(fig)
